In [ ]:
import json, re, subprocess, sys, zipfile, io
from collections import defaultdict
from pathlib import Path

def clone_repo(github_url, dest="cloned_repo"):
    dest_path = Path(dest)
    if dest_path.exists():
        print(f"{dest_path} already exists, skipping clone.")
        return dest_path
    result = subprocess.run(["git", "clone", "--depth", "1", github_url, str(dest_path)],
                             capture_output=True, text=True)
    if result.returncode != 0:
        print("CLONE FAILED:", result.stderr); sys.exit(1)
    print(f"Cloned {github_url} -> {dest_path}")
    return dest_path

In [ ]:
def find_gradle_modules(repo_dir):
    files = list(repo_dir.rglob("build.gradle")) + list(repo_dir.rglob("build.gradle.kts"))
    return [f for f in files if "/.gradle/" not in str(f) and "/build/" not in str(f)]

DEP_LINE = re.compile(
    r"""(implementation|api|compileOnly|runtimeOnly|annotationProcessor|
         testImplementation|testCompile|testRuntime|compile|runtime)
        \s*\(?\s*['"](?P<coord>[^'"]+)['"]""", re.VERBOSE)

def parse_gradle_file(path):
    text = path.read_text()
    deps = []
    for m in DEP_LINE.finditer(text):
        parts = m.group("coord").split(":")
        if len(parts) == 3:
            group, artifact, version = parts
        elif len(parts) == 2:
            group, artifact = parts; version = "MANAGED"
        else:
            continue
        deps.append({"scope": m.group(1), "group": group, "artifact": artifact,
                     "version": version, "coordinate": f"{group}:{artifact}"})
    return deps

def parse_all_modules(repo_dir):
    modules = find_gradle_modules(repo_dir)
    print(f"Found {len(modules)} Gradle build file(s):")
    all_deps = []
    for mod in modules:
        print(f"  - {mod.relative_to(repo_dir)}")
        all_deps.extend(parse_gradle_file(mod))
    return all_deps

In [ ]:
repo_dir = clone_repo("https://github.com/your-org/your-service.git")
declared = parse_all_modules(repo_dir)
declared_coords = {d["coordinate"] for d in declared}
print(f"\nDeclared dependencies: {len(declared_coords)}")
for c in sorted(declared_coords): print(f"  - {c}")

In [ ]:
def find_boot_jar(repo_dir):
    libs_dir = Path(repo_dir) / "build" / "libs"
    if not libs_dir.exists():
        return None
    candidates = [j for j in libs_dir.glob("*.jar") if "-plain" not in j.name and "sources" not in j.name]
    return candidates[0] if candidates else None

def build_class_index_from_boot_jar(boot_jar_path):
    class_index = {}
    with zipfile.ZipFile(boot_jar_path) as outer:
        lib_entries = [n for n in outer.namelist() if n.startswith("BOOT-INF/lib/") and n.endswith(".jar")]
        print(f"Found {len(lib_entries)} bundled dependency jars in {boot_jar_path.name}")
        for entry_name in lib_entries:
            jar_filename = entry_name.split("/")[-1]
            inner_bytes = outer.read(entry_name)
            m = re.match(r"^(.+?)-(\d[\w.\-]*)\.jar$", jar_filename)
            artifact = m.group(1) if m else jar_filename.replace(".jar", "")
            with zipfile.ZipFile(io.BytesIO(inner_bytes)) as inner:
                for name in inner.namelist():
                    if name.endswith(".class") and "/" in name and "module-info" not in name:
                        class_index[name[:-6].replace("/", ".")] = artifact
    return class_index

def resolve_groups_from_declared(class_index, declared_coords):
    artifact_to_full = {coord.split(":")[1]: coord for coord in declared_coords}
    return {cls: artifact_to_full.get(artifact, f"UNKNOWN_GROUP:{artifact}")
            for cls, artifact in class_index.items()}

In [ ]:
IMPORT_TO_COORDINATE = {
    "com.google.common": "com.google.guava:guava",
    "org.apache.commons.lang3": "org.apache.commons:commons-lang3",
    "commons-lang": "commons-lang:commons-lang",
    "com.fasterxml.jackson": "com.fasterxml.jackson.core:jackson-databind",
    "org.codehaus.jackson": "org.codehaus.jackson:jackson-mapper-asl",
    "com.google.gson": "com.google.code.gson:gson",
    "org.apache.commons.codec": "commons-codec:commons-codec",
    "org.slf4j": "org.slf4j:slf4j-api",
    "org.joda.time": "joda-time:joda-time",
    # Extend this as you find more duplicate-functionality candidates.
}

boot_jar = find_boot_jar(repo_dir)
if boot_jar:
    print(f"Using boot jar: {boot_jar}")
    raw_index = build_class_index_from_boot_jar(boot_jar)
    class_index = resolve_groups_from_declared(raw_index, declared_coords)
else:
    print("No boot jar in build/libs -- normal right after a fresh clone "
          "(build/ is gitignored). Run `./gradlew bootJar` for exact "
          "ground-truth detection, or proceed now with the less-precise "
          "prefix-table fallback below.")
    class_index = None

In [ ]:
CATEGORY = {
    "com.google.guava:guava": "string-utils", "org.apache.commons:commons-lang3": "string-utils",
    "commons-lang:commons-lang": "string-utils",
    "com.fasterxml.jackson.core:jackson-databind": "json", "org.codehaus.jackson:jackson-mapper-asl": "json",
    "com.google.code.gson:gson": "json", "commons-codec:commons-codec": "encoding",
    "org.slf4j:slf4j-api": "logging", "joda-time:joda-time": "datetime", "jdk:java.time": "datetime",
}
IMPORT_RE = re.compile(r"^\s*import\s+(?:static\s+)?([\w.]+)\s*;", re.MULTILINE)
CALL_RE = re.compile(r"\b([A-Za-z_][A-Za-z0-9_]*)\.([a-zA-Z0-9_]+)\s*\(")
VAR_DECL_RE = re.compile(r"\b([A-Z][A-Za-z0-9_]*)\s+([a-zA-Z_][A-Za-z0-9_]*)\s*[=;,)]")

def scan_source(repo_dir, class_index=None):
    files = [f for f in list(Path(repo_dir).rglob("*.java")) + list(Path(repo_dir).rglob("*.groovy"))
             + list(Path(repo_dir).rglob("*.kt")) if "/build/" not in str(f)]
    print(f"Scanning {len(files)} source file(s)")
    usage_count, usage_calls = defaultdict(int), defaultdict(set)
    java_time_used = False
    for f in files:
        text = f.read_text()
        imports, calls, var_decls = IMPORT_RE.findall(text), CALL_RE.findall(text), VAR_DECL_RE.findall(text)
        class_to_coord = {}
        for imp in imports:
            if imp.startswith("java.time"): java_time_used = True
            class_name = imp.split(".")[-1]
            if class_index is not None:
                coord = class_index.get(imp)
                if coord and not coord.startswith("UNKNOWN_GROUP"):
                    class_to_coord[class_name] = coord; usage_count[coord] += 1
            else:
                for root, coord in IMPORT_TO_COORDINATE.items():
                    if imp.startswith(root):
                        class_to_coord[class_name] = coord; usage_count[coord] += 1
        var_to_class = {var: cls for cls, var in var_decls if cls in class_to_coord}
        for identifier, method in calls:
            if identifier in class_to_coord:
                usage_calls[class_to_coord[identifier]].add(f"{identifier}.{method}")
            elif identifier in var_to_class:
                cls = var_to_class[identifier]
                usage_calls[class_to_coord[cls]].add(f"{cls}.{method}")
    return usage_count, usage_calls, java_time_used

usage_count, usage_calls, java_time_used = scan_source(repo_dir, class_index)

In [ ]:
used = sorted(set(usage_count.keys()) & declared_coords)
clusters = defaultdict(list)
for coord in used:
    clusters[CATEGORY.get(coord, "uncategorized")].append(coord)
if java_time_used:
    clusters["datetime"].append("jdk:java.time")
duplicate_clusters = {c: v for c, v in clusters.items() if len(v) > 1}

print(f"USED ({len(used)}): {used}")
print(f"\nDUPLICATE-FUNCTIONALITY CLUSTERS (for human review):")
for cat, coords in duplicate_clusters.items():
    print(f"  [{cat}] {coords}")
print(f"\nMETHOD CALLS OBSERVED:\n{json.dumps({k: sorted(v) for k,v in usage_calls.items()}, indent=2)}")